In [ ]:
# ==============================================================================
# IMPORTS AND CONFIGURATION
# ==============================================================================

# ------------------------------------------------------------------------------
# Standard Library Imports
# ------------------------------------------------------------------------------
import sys
import gc
import re
import os
import json
from pathlib import Path
from collections import Counter

# ------------------------------------------------------------------------------
# Append Subfolder to Path for Local Modules
# ------------------------------------------------------------------------------
sys.path.append("../../src/")  # Adjust the path if your modules are in a different folder

# ------------------------------------------------------------------------------
# Local Module Imports
# ------------------------------------------------------------------------------
import functions  # Local module (e.g., functions.py inside src/)
from functions import *  # Import all functions from file.py

# ------------------------------------------------------------------------------
# Third-Party Library Imports
# ------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------------------------
# PyTorch Imports
# ------------------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch import topk
from torch.utils.data import TensorDataset, DataLoader

# ------------------------------------------------------------------------------
# Scikit-learn Imports
# ------------------------------------------------------------------------------
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# ------------------------------------------------------------------------------
# adabmDCA Package Imports
# ------------------------------------------------------------------------------
from adabmDCA.fasta import get_tokens, write_fasta, import_from_fasta, encode_sequence, compute_weights
from adabmDCA.utils import get_device, get_dtype
from adabmDCA.stats import get_freq_single_point, get_freq_two_points, get_correlation_two_points
from adabmDCA.dataset import DatasetDCA
from adabmDCA.functional import one_hot
from adabmDCA.statmech import compute_energy
from adabmDCA.io import load_params

# ------------------------------------------------------------------------------
# arDCA Package Imports
# ------------------------------------------------------------------------------
import arDCA_paths
from arDCA_paths import arDCA_paths
from arDCA_paths.models import energy_third, energy_third_conditioned_first

# ------------------------------------------------------------------------------
# Transformer Package Imports
# ------------------------------------------------------------------------------
# from ProteinsDataset import ProteinTranslationDataset
# from ProteinTransformer import *

# ------------------------------------------------------------------------------
# Device and Data Type Settings
# ------------------------------------------------------------------------------
device = get_device("cpu")
device2 = get_device("cuda")
dtype = get_dtype("float32")

# ------------------------------------------------------------------------------
# Matplotlib Configuration
# ------------------------------------------------------------------------------
plt.rcParams.update({
    "text.usetex": True,
})


In [ ]:
def compute_CDE_full_batch_fast(sequences, model, device=device, dtype=dtype, chunk_size=150):  # Compute conservation diversity entropy for sequences using model
    with torch.no_grad():  # Disable gradient computation for efficiency
        batch_size, L, q = sequences.shape  # Unpack sequence dimensions
        CDE_full_batch = torch.zeros((batch_size, L), device=device, dtype=dtype)  # Initialize output tensor for CDE values
 
        # chunk_size = 250  # Process in chunks to manage memory
        for start in range(0, batch_size, chunk_size):  # Iterate over chunks
            end = min(start + chunk_size, batch_size)  # End index for chunk
            sequences_chunk = sequences[start:end]  # Extract current chunk
            chunk_batch_size = sequences_chunk.shape[0]  # Size of current chunk
            model_device = {}  # Move model to device
            model_device["bias"] = model["bias"].to(device)  # Bias to device
            model_device["coupling_matrix"] = model["coupling_matrix"].to(device)  # Coupling matrix to device
            sequence_indices = sequences_chunk.argmax(-1)  # Get amino acid indices
            base = sequence_indices.unsqueeze(1).expand(-1, L * q, -1).reshape(chunk_batch_size * L * q, L).clone().to(device)  # Base sequences for mutations
            positions = torch.arange(L, device=device).repeat_interleave(q).repeat(chunk_batch_size)  # Positions for mutations
            values = torch.arange(q, device=device, dtype=torch.int64).repeat(L).repeat(chunk_batch_size)  # Amino acid values for mutations
            base[torch.arange(chunk_batch_size * L * q, device=device), positions] = values  # Set mutations in base
            dms = base  # Deep mutational scan sequences
            dms_oh = one_hot(dms, num_classes=q).to(dtype)  # One-hot encode DMS sequences
            energy_dms = compute_energy(dms_oh, model_device)  # Compute energies for DMS
            energy_dms = energy_dms.view(chunk_batch_size, L, q)  # Reshape to (batch, L, q)
            energy_dms -= energy_dms.min(dim=2, keepdim=True)[0]  # Normalize energies
            exp_energy_dms = torch.exp(-energy_dms)  # Exponentiate negative energies
            sums = exp_energy_dms.sum(dim=2, keepdim=True)  # Sum over amino acids
            p_i = exp_energy_dms / sums  # Probabilities
            CDE_chunk = -torch.sum(p_i * torch.log2(p_i), dim=2)  # Compute entropy (CDE)
            CDE_full_batch[start:end] = CDE_chunk.cpu()  # Store chunk result
            del base, positions, values, dms, dms_oh, energy_dms, exp_energy_dms, sums, p_i, model_device["bias"], model_device["coupling_matrix"], CDE_chunk  # Free memory
            
        # Clean memory  # Additional cleanup
        del sequences  # Delete input sequences
        gc.collect()  # Garbage collect
        torch.cuda.empty_cache()  # Clear CUDA cache
        
    return CDE_full_batch.cpu()  # Return CDE values on CPU

# Data Loading and Preparation

## Load Evolutionary Sequence Data

This section loads protein sequences from different evolutionary timescales. Each timescale represents a different amount of evolutionary time, allowing us to study how sequences diverge over time.

**Key steps:**
- Define amino acid alphabet and timescales to analyze
- Load sequences from FASTA files for each timescale
- Sample 20,000 sequences from each timescale for analysis
- Prepare datasets for downstream processing


In [ ]:
protein_family = "PF00072" #"Chorismate Mutase" #"PF00072" # "betalactamase"

if protein_family == "Chorismate Mutase":
    data_path = "generated_data/CM/train_validation" #"generated_data/CM"
    MSA_path = "MSAs/CM_130530_MC.fasta"
    original_model_path = "evolution_bmDCA_model/CM/params.dat"
    t = 0
    # model_paths = "predict_Time/LR_trained/CM/"
    output_path = "immagini_paper/CM/train_validation/"
    cde_path = "generated_data/CM/train_validation/full_cde_train"

elif protein_family == "PF00072":
    data_path = "generated_data/PF00072/train_validation"
    MSA_path = "MSAs/PF00072.fasta"
    original_model_path = "evolution_bmDCA_model/PF00072/params.dat"
    t = 0
    model_paths = "models/PF00072/"
    output_path = "immagini_paper/PF00072/train_validation/"
    cde_path = "generated_data/PF00072/train_validation/full_cde_train"

elif protein_family == "betalactamase":
    data_path = "generated_data/betalactamase/train_validation"
    MSA_path = "MSAs/betalactamase_nodupl_natural_noflankgaps.fa"
    original_model_path = "evolution_bmDCA_model/betalactamase/params.dat"
    t = 0
    model_paths = "models/betalactamase/"
    output_path = "immagini_paper/betalactamase/predict_time/"
    reg = "_rJ1e-4_rH1e-6"
    cde_path = "generated_data/betalactamase/train_validation/full_cde_train"

In [ ]:
data_type = "ACDEFGHIKLMNPQRSTVWY-"
tokens = get_tokens(data_type)

timescales = ["10e1", "10e2", "10e3", "10e4", "10e5", "10e6"] 
n_seqs = 30_000

data_files = [
    f"{data_path}/10e1_train.fasta",
    f"{data_path}/10e2_train.fasta",
    f"{data_path}/10e3_train.fasta",
    f"{data_path}/10e4_train.fasta",
    f"{data_path}/10e5_train.fasta",
    f"{data_path}/10e6_train.fasta",
]

datasets = {}
for t, path in zip(timescales, data_files):
    ds = DatasetDCA(path_data=path, alphabet=data_type, device=device, dtype=dtype, no_reweighting=True)
    # indices = torch.randperm(ds.data.shape[0])[:n_seqs]
    datasets[t] = ds.data[:n_seqs]#.argmax(dim=-1)
    del ds
    gc.collect()

print(f"Loaded {len(datasets)} datasets with {n_seqs} sequences each")


datasets_oh = {
    key: one_hot(data, num_classes=21).to(dtype=dtype)
    for key, data in datasets.items()
}

## Load Natural MSA and bmDCA Model

This section imports the natural Multiple Sequence Alignment (MSA) and loads the pre-trained Boltzmann machine Direct Coupling Analysis (bmDCA) model.

**Purpose:**
- The natural MSA provides reference sequences from real biological data
- The bmDCA model captures evolutionary constraints and amino acid couplings
- These will be used to compute conservation scores (CDE) for sequence positions


In [ ]:
# Get number of amino acid classes from first dataset
q = datasets_oh[timescales[0]].shape[2] 
print("q =",q)

# Import natural MSA from FASTA file
headers, msa_enc_nat = import_from_fasta(MSA_path , tokens=tokens, filter_sequences=True)
# Convert to one-hot encoding
msa_oh_nat = one_hot(torch.tensor(msa_enc_nat, device=device, dtype=torch.int32), num_classes=q).to(dtype)

# Get dataset dimensions: M samples, L3 sequence length (3*L for triplets), q amino acids
M, L3 = datasets[timescales[0]].shape
L = L3 // 3  # Each sequence is split into 3 parts of length L

# Load bmDCA model parameters from file
print("Loading bmDCA model from:", original_model_path)
original_tokens = get_tokens("ACDEFGHIKLMNPQRSTVWY-")
original_model = load_params(original_model_path, tokens=original_tokens, device=device, dtype=dtype)

# Feature Extraction

## Compute Conservation Diversity Entropy (CDE)

Conservation Diversity Entropy (CDE) measures the evolutionary constraints at each position in a protein sequence. Lower CDE indicates higher conservation (more constrained positions), while higher CDE indicates more variability.

**Computed for:**
- **Time 0**: Initial sequences (first third of triplets)
- **Time T**: Evolved sequences (second third of triplets)

**Output:**
- Full CDE vectors: per-position conservation scores (96 features per sequence)
- Mean CDE: average conservation across all positions (1 feature per sequence)


In [ ]:
# Initialize dictionaries to store CDE values for all timescales
CDE_full_0 = {}  # Full CDE per position at time 0
CDE_mean_0 = {}  # Mean CDE across positions at time 0
CDE_full_T  = {}  # Full CDE per position at time T
CDE_mean_T  = {}  # Mean CDE across positions at time T

# Load precomputed CDE for Chorismate Mutase
if protein_family == "Chorismate Mutase" or protein_family == "PF00076" or protein_family == "PF00072_2nd" or protein_family == "betalactamase":
    cde_dir = Path(cde_path)
    if not cde_dir.exists():
        raise FileNotFoundError(f"CDE directory not found: {cde_dir}")

    loaded_CDE_0 = np.load(cde_dir / "CDE_0.npy", allow_pickle=True).item()
    loaded_CDE_0_mean = np.load(cde_dir / "CDE_0_mean.npy", allow_pickle=True).item()
    loaded_CDE_T = np.load(cde_dir / "CDE_T.npy", allow_pickle=True).item()
    loaded_CDE_T_mean = np.load(cde_dir / "CDE_T_mean.npy", allow_pickle=True).item()

    for key in timescales:
        if key not in loaded_CDE_0 or key not in loaded_CDE_0_mean or key not in loaded_CDE_T or key not in loaded_CDE_T_mean:
            raise KeyError(f"Missing key '{key}' in precomputed CDE files")

        # Load only the first n_seqs values, not the full arrays
        cde_0_key = np.asarray(loaded_CDE_0[key])[:n_seqs]
        cde_t_key = np.asarray(loaded_CDE_T[key])[:n_seqs]
        cde_0_mean_key = np.asarray(loaded_CDE_0_mean[key])[:n_seqs]
        cde_t_mean_key = np.asarray(loaded_CDE_T_mean[key])[:n_seqs]

        # Keep full CDE as torch tensors for downstream indexing
        CDE_full_0[key] = torch.as_tensor(cde_0_key, dtype=dtype)
        CDE_full_T[key] = torch.as_tensor(cde_t_key, dtype=dtype)

        # Keep mean CDE as numpy arrays to match downstream code
        CDE_mean_0[key] = cde_0_mean_key
        CDE_mean_T[key] = cde_t_mean_key

    print(f"Loaded precomputed CDE from: {cde_dir} (first {n_seqs} entries per timescale)")
else:
    raise ValueError(
        f"Precomputed CDE path not configured for protein_family={protein_family}. "
        "Set cde_path before running this cell."
    )


# ------------------------------------------------------------------------------
# Old CDE computation code kept for reference (commented as requested)
# ------------------------------------------------------------------------------
# for key in timescales:

#     dataset_CDE = datasets_oh[key] #one_hot(datasets[key], num_classes=21).to(dtype)
#     print(f"Computing CDE for {key}...")

#     # Compute CDE for first third of sequence (time 0)
#     CDE_full_0[key] = compute_CDE_full_batch_fast(dataset_CDE[:, :L, :],   original_model, device=device2, dtype=dtype, chunk_size=150).cpu()
#     # Compute CDE for second third of sequence (time T)
#     CDE_full_T[key]  = compute_CDE_full_batch_fast(dataset_CDE[:, L:2*L, :], original_model, device=device2, dtype=dtype, chunk_size=150).cpu()
#     # Compute mean CDE along sequence length dimension
#     CDE_mean_0[key] = torch.mean(CDE_full_0[key], dim=1).numpy()
#     CDE_mean_T[key]  = torch.mean(CDE_full_T[key],  dim=1).numpy()

#     # Clean up memory
#     del dataset_CDE
#     gc.collect()

## Compute Sequence Distances

Calculate the evolutionary distance between sequences at time 0 and time T using Hamming distance metrics.

**Two distance metrics computed:**

1. **Total Hamming Distance**: Total number of mismatches between time 0 and time T sequences
2. **Position-wise Mismatch Vector**: Binary vector (96 features) indicating match/mismatch at each position
   - 0 = amino acid unchanged
   - 1 = amino acid mutated

These features capture the spatial pattern of mutations across the sequence.


In [ ]:
def compute_distance(sample):
    """
    Compute total Hamming distance between first and second thirds of sequence
    Args:
        sample: tensor of shape (batch_size, 3*L) representing triplets
    Returns:
        tuple containing tensor of distances for each sample
    """
    l = sample.shape[1] // 3
    a, b, c = sample.split(l, dim=1)
    return (a != b).sum(dim=1), 

print("Computing data distances...")  # Notify start of distance computation
# Calculate total Hamming distance for each sequence in each timescale
distance = {  # Dict of distances per key, if key in timescales
    key: compute_distance(data) # For each sample in dataset
    for key, data in datasets.items() if key in str(timescales)}  # Filter datasets by timescales


def compute_distance_vector(sample):
    """
    Compute binary mismatch vector between first and second thirds
    Args:
        sample: tensor of shape (batch_size, 3*L) representing triplets
    Returns:
        Binary vector: 1 if mismatch, 0 if match at each position
    """
    l = sample.shape[1] // 3
    a, b, c = sample.split(l, dim=1)
    # Returns binary vector: 1 if mismatch, 0 if match
    # Since sample is already indices (B, L), not one-hot
    mismatch_vector = (a != b).float()  # (B, l)
    return mismatch_vector

print("Computing data distance vectors...")  # Notify start of distance computation
# Calculate position-wise mismatch vectors for each sequence
distance_vector = {  # Dict of mismatch vectors per key
    key: compute_distance_vector(data)  # Binary vector for each sample
    for key, data in datasets.items() if key in str(timescales)}  # Filter datasets by timescales

## Train/Test Split

Divide all datasets and computed features into training and test sets to enable proper model evaluation.

**Split ratio:** 80% training / 20% testing

**Data split includes:**
- Original sequence datasets
- Distance vectors (position-wise mismatches)
- CDE features (both full vectors and means for time 0 and time T)
- Total Hamming distances

All splits use the same random indices to ensure consistency across all feature types.


In [ ]:
# Initialize dictionaries for train and test splits
datasets_train = {}
datasets_test = {}
distance_vector_train = {}
distance_vector_test = {}
CDE_0_train = {}
CDE_0_test = {}
CDE_0_mean_train = {}
CDE_0_mean_test = {}
CDE_T_train = {}
CDE_T_test = {}
CDE_T_mean_train = {}
CDE_T_mean_test = {}
distance_train = {}
distance_test = {}
indices_train = {}
indices_test = {}

# Define train/test split ratio
train_ratio = 0.8

# Split each timescale dataset into train and test sets
for t in timescales:
    n_samples = datasets[t].shape[0]
    # Generate random permutation of indices for splitting
    indices = torch.randperm(n_samples)
    
    # Calculate split point
    split_idx = int(n_samples * train_ratio)
    
    # Split indices into train and test
    train_idx = indices[:split_idx]
    test_idx = indices[split_idx:]
    
    # Store split datasets
    datasets_train[t] = datasets_oh[t][train_idx]
    datasets_test[t] = datasets_oh[t][test_idx]
    
    # Split CDE data using the same indices
    CDE_0_train[t] = CDE_full_0[t][train_idx]
    CDE_0_test[t] = CDE_full_0[t][test_idx]
    CDE_0_mean_train[t] = CDE_mean_0[t][train_idx]
    CDE_0_mean_test[t] = CDE_mean_0[t][test_idx]
    CDE_T_train[t] = CDE_full_T[t][train_idx]
    CDE_T_test[t] = CDE_full_T[t][test_idx]
    CDE_T_mean_train[t] = CDE_mean_T[t][train_idx]
    CDE_T_mean_test[t] = CDE_mean_T[t][test_idx]
    
    # Split distance data using the same indices
    # distance[t] is a tuple, take the first element
    distance_train[t] = distance[t][0][train_idx]
    distance_test[t] = distance[t][0][test_idx]
    # Split distance vector data using the same indices
    distance_vector_train[t] = distance_vector[t][train_idx]
    distance_vector_test[t] = distance_vector[t][test_idx]
    
    # Store indices for reference
    indices_train[t] = train_idx
    indices_test[t] = test_idx
    
    print(f"{t}: {len(train_idx)} training samples, {len(test_idx)} test samples")

In [ ]:
# Create output directory if it doesn't exist
os.makedirs(output_path, exist_ok=True)

# Logistic Regression Training

## Train LR Models with Different Feature Combinations

Train Logistic Regression classifiers to predict evolutionary timescales using various combinations of features.

**Feature combinations tested:**
1. **Matched positions sum only**: Sum of binary mismatch vector (1 feature)
2. **Matched positions sum + CDE_0 mean**: Mismatch count + average conservation at time 0 (2 features)
3. **Matched positions sum + CDE_T mean**: Mismatch count + average conservation at time T (2 features)
4. **Matched positions only**: Binary mismatch vector (96 features)
5. **Matched positions + CDE_0 mean**: Distance + average conservation at time 0 (97 features)
6. **Matched positions + both CDE means**: Distance + both time point averages (98 features)
7. **Matched positions + full CDE_0**: Distance + per-position conservation at time 0 (192 features)
8. **All features**: Distance + full CDE vectors for both time points (288 features)

**Goal:** Determine which features are most informative for predicting evolutionary timescale.


In [ ]:
L

In [ ]:
# ==========================================
# SELECT SPECIFIC TIMESCALES
# ==========================================
selected_timescales = timescales

# ==========================================
# PREPARE DATA WITH DISTANCE VECTOR + FULL CDE VECTORS + MEANS (MEMORY EFFICIENT)
# ==========================================
print(f"Preparing data with distance vector ({L}) + full CDE vectors ({L}+{L}) + means - Memory efficient...")

# Calculate total number of samples across all timescales
total_train = sum(datasets_train[t].shape[0] for t in selected_timescales)
total_test = sum(datasets_test[t].shape[0] for t in selected_timescales)

print(f"Total training samples: {total_train}")
print(f"Total test samples: {total_test}")

# Pre-allocate arrays (96 distance + 96 CDE_0 + 96 CDE_T + 1 CDE_0_mean + 1 CDE_T_mean = 290 features)
n_features = L + L + L + 1 + 1
X_train_full = np.zeros((total_train, n_features), dtype=np.float32)
X_test_full = np.zeros((total_test, n_features), dtype=np.float32)
y_train_full = []
y_test_full = []

# Fill training data by concatenating all timescales
idx_train = 0
for t in selected_timescales:
    n_samples = datasets_train[t].shape[0]
    
    # Distance vector (96 features) - binary mismatch indicators
    dist_vectors = distance_vector_train[t].cpu().numpy()
    X_train_full[idx_train:idx_train+n_samples, 0:L] = dist_vectors
    
    # CDE_0 (96 features) - conservation at time 0
    for i in range(n_samples):
        X_train_full[idx_train+i, L:2*L] = CDE_0_train[t][i].flatten()
    
    # CDE_T (96 features) - conservation at time T
    for i in range(n_samples):
        X_train_full[idx_train+i, 2*L:3*L] = CDE_T_train[t][i].flatten()
    
    # CDE_0_mean (1 feature) - average conservation at time 0
    X_train_full[idx_train:idx_train+n_samples, 3*L] = CDE_0_mean_train[t]
    
    # CDE_T_mean (1 feature) - average conservation at time T
    X_train_full[idx_train:idx_train+n_samples, 3*L+1] = CDE_T_mean_train[t]
    
    # Labels - timescale identifier
    y_train_full.extend([t] * n_samples)
    
    idx_train += n_samples
    del dist_vectors
    gc.collect()

print(f"Training data prepared: {X_train_full.shape}")

# Fill test data with same feature structure
idx_test = 0
for t in selected_timescales:
    n_samples = datasets_test[t].shape[0]
    
    # Distance vector (96 features)
    dist_vectors = distance_vector_test[t].cpu().numpy()
    X_test_full[idx_test:idx_test+n_samples, 0:L] = dist_vectors
    
    # CDE_0 (96 features)
    for i in range(n_samples):
        X_test_full[idx_test+i, L:2*L] = CDE_0_test[t][i].flatten()
    
    # CDE_T (96 features)
    for i in range(n_samples):
        X_test_full[idx_test+i, 2*L:3*L] = CDE_T_test[t][i].flatten()
    
    # CDE_0_mean (1 feature)
    X_test_full[idx_test:idx_test+n_samples, 3*L] = CDE_0_mean_test[t]
    
    # CDE_T_mean (1 feature)
    X_test_full[idx_test:idx_test+n_samples, 3*L+1] = CDE_T_mean_test[t]
    
    # Labels
    y_test_full.extend([t] * n_samples)
    
    idx_test += n_samples
    del dist_vectors
    gc.collect()

print(f"Test data prepared: {X_test_full.shape}")

# Create DataFrames with descriptive column names
dist_cols = [f'dist_{j}' for j in range(L)]
cde_0_cols = [f'CDE_0_{j}' for j in range(L)]
cde_t_cols = [f'CDE_T_{j}' for j in range(L)]
feature_names = dist_cols + cde_0_cols + cde_t_cols + ['CDE_0_mean', 'CDE_T_mean']

df_train = pd.DataFrame(X_train_full, columns=feature_names)
df_train['matched_positions_sum'] = df_train[dist_cols].sum(axis=1)
df_train['timescale'] = y_train_full

df_test = pd.DataFrame(X_test_full, columns=feature_names)
df_test['matched_positions_sum'] = df_test[dist_cols].sum(axis=1)
df_test['timescale'] = y_test_full

print(f"Training DataFrame shape: {df_train.shape}")
print(f"Test DataFrame shape: {df_test.shape}")

# Clean up large arrays to free memory
del X_train_full, X_test_full, y_train_full, y_test_full
gc.collect()

# ==========================================
# TIMESCALE SORTING AND ENCODING
# ==========================================
def parse_timescale(ts):
    """
    Parse timescale string and convert to numerical value
    Example: '1x10e2' -> 1 * 10^2 = 100
    """
    m = re.match(r'(\d+)e(\d+)', ts)

    base, exp = int(m.group(1)), int(m.group(2))
    print(f"Parsing timescale: {ts} -> base: {base}, exp: {exp}")
    return (base ** exp)

# Sort timescales by numerical value and create class encoding
sorted_timescales = sorted(df_train['timescale'].unique(), key=parse_timescale)
timescale_to_index = {ts: i for i, ts in enumerate(sorted_timescales)}

# Convert timescale labels to class indices
y_train_class = df_train['timescale'].map(timescale_to_index).values
y_test_class = df_test['timescale'].map(timescale_to_index).values

# ==========================================
# TEST DIFFERENT FEATURE COMBINATIONS
# ==========================================
matched_positions_sum_cols = ['matched_positions_sum']

feature_combinations = [
    (matched_positions_sum_cols, 'matched_positions_sum'),
    (matched_positions_sum_cols + ['CDE_0_mean'], 'matched_positions_sum_CDE0_mean'),
    (matched_positions_sum_cols + ['CDE_T_mean'], 'matched_positions_sum_CDET_mean'),
    (dist_cols, 'matched_positions'),
    (dist_cols + ['CDE_0_mean'], 'matched_positions_CDE0_mean'),
    (dist_cols + ['CDE_0_mean', 'CDE_T_mean'], 'matched_positions_CDE0_mean_CDET_mean'),
    (dist_cols + cde_0_cols, 'matched_positions_CDE0'),
    (dist_cols + cde_0_cols + cde_t_cols, 'matched_positions_CDE0_CDET_full'),
]

# ==========================================
# LOGISTIC REGRESSION
# ==========================================
print("\n" + "="*70)
print("LOGISTIC REGRESSION - Classification")
print("="*70)

results_lr = {}

# Train logistic regression model for each feature combination
for features, name in feature_combinations:
    print("\n" + "="*70)
    print(f"TESTING LOGISTIC REGRESSION WITH FEATURES: {name}")
    print(f"Number of features: {len(features)}")
    print("="*70)
    
    # Prepare features from DataFrame
    X_train = df_train[features].values
    X_test = df_test[features].values
    
    # Standardize features (zero mean, unit variance)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train logistic regression with multinomial classification
    log_reg = LogisticRegression(max_iter=2000, random_state=42, multi_class='multinomial', solver='lbfgs')
    log_reg.fit(X_train_scaled, y_train_class)
    
    # Make predictions on test set
    y_pred_class = log_reg.predict(X_test_scaled)
    
    # Evaluate model performance
    accuracy = accuracy_score(y_test_class, y_pred_class)
    print(f"\nTest Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test_class, y_pred_class, target_names=sorted_timescales))
    
    # Store results for later comparison
    results_lr[name] = {
        'accuracy': accuracy,
        'model': log_reg,
        'scaler': scaler,
        'features': features,
        'n_features': len(features)
    }
    
    # Clean up memory
    del X_train, X_test, X_train_scaled, X_test_scaled
    gc.collect()


In [ ]:
# import joblib
# # Save all trained LR models to disk for later use
# os.makedirs(output_path, exist_ok=True)
# for name, result in results_lr.items():
#     model_path = os.path.join(output_path, f"LR_model_{name}_{n_seqs}_chains.joblib")
#     # Save model, scaler, and feature list together
#     joblib.dump({
#         'model': result['model'],
#         'scaler': result['scaler'],
#         'features': result['features']
#     }, model_path)
#     print(f"Saved LR model with features '{name}' to {model_path}")